# `reducnn` CIFAR-100 Research Suite (GitHub Version) 🔬

This notebook executes structural pruning experiments on the **CIFAR-100** dataset using the `reducnn` framework. It is optimized for GitHub/Colab environments.

In [ ]:
# --- STEP 0: GITHUB BOOTLOADER (Colab Optimization) ---
import sys, os

# 1. Clone the repository from GitHub
repo_url = "https://github.com/albertraviss2023/activation-based-pruning.git"
repo_dir = "activation-based-pruning"

if not os.path.exists(repo_dir):
    print(f"🚀 Cloning {repo_url}...")
    !git clone {repo_url}
else:
    print(f"✅ Repository {repo_dir} already exists. Updating...")
    %cd {repo_dir}
    !git pull
    %cd ..

# 2. Environment Setup
os.chdir(repo_dir)
sys.path.insert(0, os.path.abspath("src"))

# 3. Install dependencies and editable package
!pip install -q --upgrade setuptools pip
!pip install -e .

# 4. Load autoreload and Verify

# --- Python 3.12 Compatibility Shim ---
import sys
try:
    import imp
except ImportError:
    from types import ModuleType
    import importlib
    imp = ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp
    print("🛠️ Applied Python 3.12 'imp' shim")

%load_ext autoreload
%autoreload 2
import reducnn
print(f"\n✅ System Ready! Module loaded from: {reducnn.__file__}")

In [ ]:
import torch, torchvision, torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import reducnn as sp
import reducnn.visualization as viz
from reducnn.pruner import ReduCNNPruner
from reducnn.backends.torch_backend import PyTorchAdapter

# Local repository paths
checkpoint_dir = "my_models/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"📁 Local workspace initialized at: {os.getcwd()}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

train_set = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=128)

print(f"✅ CIFAR-100 ready with {len(train_set)} training samples.")

In [ ]:
adapter = PyTorchAdapter(config={'lr': 1e-3, 'input_shape': (3, 32, 32), 'num_classes': 100})
model = adapter.get_model("vgg16")

print("🔥 Training VGG16 Baseline on CIFAR-100...")
adapter.train(model, train_loader, epochs=5, name="VGG16_CIFAR100", val_loader=test_loader)
b_acc = adapter.evaluate(model, test_loader)
print(f"\n✅ Baseline: {b_acc:.2f}%")

In [ ]:
print("🔬 PRUNING: L1-Norm (Local 30%)")
surgeon = ReduCNNPruner(method='l1_norm', scope='local')
pruned_model, masks, duration = surgeon.prune(model, train_loader, ratio=0.3)

print("💊 Fine-tuning pruned model...")
adapter.train(pruned_model, train_loader, epochs=3, name="Heal_L1_CIFAR100", val_loader=test_loader)
p_acc = adapter.evaluate(pruned_model, test_loader)
print(f"\n✅ Pruned Accuracy: {p_acc:.2f}%")

viz.plot_layer_sensitivity(masks, "VGG16 L1-Norm Local 30% (CIFAR-100)")